# 05 — CNN smoke test: does the data/model plumbing run end-to-end and can it overfit a tiny batch?

**Decision this feeds (`structuring-ml-projects` SKILL.md, cheap-proxy
ladder, `deep-learning-imaging.md`):** rung 0 (mandatory before any
longer run) — right shapes/dtypes end to end, and the model can overfit
a tiny labeled subset to near-zero loss. If rung 0 passes, rung 1 (a
10-20% stratified subset, one fold) checks the loss curve is sane. Note:
**this notebook does not decide anything about the model's real
performance** — only rung 3 (a full CV run, not built yet) clears the
gate against the classical baseline (`model.build_combat_baseline()`,
0.5290 log loss). See
`docs/superpowers/specs/2026-09-08-cnn-data-plumbing-design.md`.

**Data handling:** this notebook loads real `.nii.gz` volumes and
row-level labels, so per the AI-assistant data rule (`README.md`) it is
**[RUN ME]** — run it yourself, share back only the printed shapes and
loss values, not any per-row output.

## QC: does the fixed crop box actually contain the striatum?

`CROP_CENTER_MM`/`CROP_SIZE_MM` (`config.py`) were measured against a
different axis convention than `data.crop_or_pad` applies them in; this
cell is a cheap sanity check, not a proof. For a few real volumes it
prints the intensity-weighted center of mass of the *cropped* array as a
fraction of that array's shape per axis (0 = one edge, 1 = the other,
0.5 = geometric center) -- a striatum-centered crop should land close to
0.5 on each axis, not pinned near 0 or 1. **[RUN ME]** -- loads real
pixel data.

In [ ]:
# [RUN ME] -- loads real pixel data (no labels needed for this check).
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import config
import data

qc_labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
qc_uids = qc_labels_df[config.UID_COLUMN].tolist()[:3]

for uid in qc_uids:
    volume = data.load_volume(uid)[0]  # drop channel dim -> (56, 30, 44)
    signal = np.clip(volume, 0, None)
    if signal.sum() <= 0:
        print(f"{uid}: no positive signal in cropped volume, cannot compute centroid")
        continue
    idx = np.indices(signal.shape, dtype=float)
    centroid_vox = [np.average(idx[axis], weights=signal) for axis in range(3)]
    centroid_frac = [c / (s - 1) for c, s in zip(centroid_vox, signal.shape)]
    print(f"{uid}: centroid as fraction of shape (0.5 = geometric center) = "
          f"{[round(f, 3) for f in centroid_frac]}")

In [ ]:
# [RUN ME] -- loads real pixel data + row-level labels.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import config
import dataset
import model

torch.manual_seed(config.SEED)

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
rng = np.random.RandomState(config.SEED)
smoke_idx = rng.choice(len(labels_df), size=24, replace=False)
smoke_df = labels_df.iloc[smoke_idx]

smoke_ds = dataset.DatParkinsonDataset(
    uids=smoke_df[config.UID_COLUMN].tolist(),
    labels=smoke_df[config.TARGET_COLUMN].tolist(),
)
loader = torch.utils.data.DataLoader(smoke_ds, batch_size=8, shuffle=True)

net = model.build_model()
opt = torch.optim.Adam(net.parameters(), lr=config.LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

print(f"rung 0: {len(smoke_ds)} labeled volumes, checking shapes + overfit")
for x, y in loader:
    print("batch shapes:", x.shape, x.dtype, y.shape, y.dtype)
    break

print("preprocessing (nibabel resample) dominates the per-epoch time, not "
      "the model itself -- printing per-epoch progress below so this "
      "doesn't look hung.")
start = time.time()
losses = []
for epoch in range(30):
    epoch_loss = 0.0
    for x, y in loader:
        opt.zero_grad()
        loss = loss_fn(net(x), y)
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * x.shape[0]
    losses.append(epoch_loss / len(smoke_ds))
    print(f"epoch {epoch}: loss = {losses[-1]:.4f} ({time.time() - start:.1f}s elapsed)")

print("rung 0 loss curve (first/mid/last):", losses[0], losses[len(losses) // 2], losses[-1])

In [ ]:
# [RUN ME] -- only run after rung 0 above shows the loss reaching near
# zero. Loads a larger real subset + labels.
import evaluate

# inplane_family isn't in train_labels.csv itself (it's derived from NIfTI
# headers) -- reuse notebooks/03's baseline_features.csv, which already
# has it per uid, rather than re-deriving it here.
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner")

rng2 = np.random.RandomState(config.SEED)
subset_idx = rng2.choice(len(labeled_df), size=int(0.15 * len(labeled_df)), replace=False)
subset_df = labeled_df.iloc[subset_idx].reset_index(drop=True)

folds = evaluate.make_folds(
    subset_df[config.TARGET_COLUMN].to_numpy(),
    subset_df["inplane_family"].to_numpy(),
    n_splits=config.N_FOLDS, random_state=config.SEED,
)
train_idx, val_idx = folds[0]

train_ds = dataset.DatParkinsonDataset(
    uids=subset_df.iloc[train_idx][config.UID_COLUMN].tolist(),
    labels=subset_df.iloc[train_idx][config.TARGET_COLUMN].tolist(),
)
val_ds = dataset.DatParkinsonDataset(
    uids=subset_df.iloc[val_idx][config.UID_COLUMN].tolist(),
    labels=subset_df.iloc[val_idx][config.TARGET_COLUMN].tolist(),
)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS,
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=config.BATCH_SIZE, num_workers=config.NUM_WORKERS,
)

net = model.build_model()
opt = torch.optim.Adam(net.parameters(), lr=config.LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

# Rung 1 is a smoke-test proxy, not a real training run -- the spec calls
# for "few epochs", not config.EPOCHS (50, the full-training default), so
# this uses its own small local epoch count instead.
rung1_epochs = 15

print(f"rung 1: train={len(train_ds)}, val={len(val_ds)}, {rung1_epochs} epochs, 1 fold")
print("this can take several minutes -- preprocessing (nibabel resample) "
      "dominates, not the model itself; it has not hung.")
start = time.time()
for epoch in range(rung1_epochs):
    net.train()
    for x, y in train_loader:
        opt.zero_grad()
        loss = loss_fn(net(x), y)
        loss.backward()
        opt.step()

    net.eval()
    val_probs, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            val_probs.append(torch.sigmoid(net(x)).numpy())
            val_labels.append(y.numpy())
    val_probs = np.concatenate(val_probs)
    val_labels = np.concatenate(val_labels)
    val_loss = evaluate.log_loss_score(val_labels, val_probs)
    print(f"epoch {epoch}: val log loss = {val_loss:.4f} "
          f"({time.time() - start:.1f}s elapsed)")

print(f"rung 1 final val log loss: {val_loss:.4f} "
      f"(reference only -- classical build_combat_baseline() = 0.5290; "
      f"this is NOT a gate decision)")

**What we're looking for:** does the plumbing run end to end with the
right shapes, and can the model overfit a tiny labeled batch (rung 0)?
If so, is the rung-1 loss curve sane (not diverging, not stuck at the
base-rate loss)?

**What we found:** *(paste the printed batch shapes, rung-0 loss curve,
and rung-1 final val log loss here after running the cells above)*

**Decision / next step:** *(if rung 0 fails to overfit, that's a bug in
data.py/dataset.py/model.py to fix before anything else. If rung 0
passes, the follow-up spec — full train.py, augmentation, rung 2/3 — is
next. Rung 1's number is a proxy only; it does not clear the gate.)*